In [1]:
from pathlib import Path
from osgeo import gdal
gdal.UseExceptions()
import numpy as np

input and output directory path

In [2]:
project_root = Path().resolve().parent.parent
# output_data_folder = project_root / "data" / "csen12" / "test_p"
# selected_rasters_path = project_root / "data" / "csen12" / "test_p" / "rasters_h_in_aoi.txt"

Define bands that will be used as features and labels and their index in data

In [3]:
csen12_feature_bands = {
    "B02": 2,
    "B03": 3,
    "B04": 4,
    "B05": 5,
    "B06": 6,
    "B07": 7,
    "B8A": 9,
}
csen12_label_bands = {"CM1":14}

Extract bands and create new dataset

In [4]:
def extract_bands(raster_paths: list[Path],output_folder: Path, bands_to_extract, features = True):
    output_folder.mkdir(parents=True,exist_ok=True)
    for tif in raster_paths:
        ds = gdal.Open(str(tif))
    
        # print(f"Processing: {tif.name}")
    
        # Create output
        if features:
            output_file = output_folder / f"{tif.stem}_image.tif"
        else:
            output_file = output_folder / f"{tif.stem}_label.tif"
        driver = gdal.GetDriverByName("GTiff")
        out = driver.Create(
            str(output_file),
            ds.RasterXSize,
            ds.RasterYSize,
            len(bands_to_extract),
            gdal.GDT_UInt16,
            ["COMPRESS=LZW"]
        )
    
        out.SetProjection(ds.GetProjection())
        out.SetGeoTransform(ds.GetGeoTransform())
    
        for idx, (band_name, band_number) in enumerate(bands_to_extract.items(), start=1):
            band = ds.GetRasterBand(band_number)
            data = band.ReadAsArray()
            if not features:
                # combine thin (2) and thick (1) clouds into clouds (1)
                data = np.where(data==2,1,data)
                # change value for cloud shadow from 3 to 2
                data = np.where(data==3,2,data)
            out_band = out.GetRasterBand(idx)
            out_band.WriteArray(data)
            out_band.SetDescription(band_name)
            # print(f" Extracted {band_name} (Band {band_number})")

        out.FlushCache()
        out = None
        ds = None

        print(f" Saved: {output_file.name}\n")
        

In [6]:
selected_rasters_path = project_root / "data" / "csen12" / "preprocess_test" / "rasters_h_in_aoi_test.txt"
selected_rasters = [
    Path(line.strip())
    for line in selected_rasters_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
output_data_folder = project_root / "data" / "csen12" / "preprocess_test" / "training_dataset_original"

In [7]:
# selected_rasters = []
# input_data_folder = project_root / "data" / "csen12" / "preprocess_test"
# output_data_folder = project_root / "data" / "csen12" / "test_p" / "original"
# for raster_path in input_data_folder.glob("*.tif"):
#     selected_rasters.append(raster_path)

In [8]:
extract_bands(selected_rasters, output_data_folder, csen12_feature_bands,features=True)
extract_bands(selected_rasters, output_data_folder,csen12_label_bands, features=False)

 Saved: 20190220T072939_20190220T074111_T38RPR_image.tif
 Saved: 20191018T072909_20191018T073856_T38RPS_image.tif

 Saved: 20191112T073141_20191112T073144_T38RPS_image.tif
 Saved: 20200316T072639_20200316T073813_T38RPS_image.tif

 Saved: 20200415T072609_20200415T073422_T38RPR_image.tif
 Saved: 20181030T081041_20181030T082011_T37SDU_image.tif

 Saved: 20190324T080609_20190324T081236_T37SDU_image.tif
 Saved: 20190329T080611_20190329T081207_T37SDU_image.tif

 Saved: 20190423T080619_20190423T081724_T37SDU_image.tif
 Saved: 20191109T081049_20191109T081833_T37SDU_image.tif

 Saved: 20190220T072939_20190220T074111_T38RPR_label.tif

 Saved: 20191018T072909_20191018T073856_T38RPS_label.tif

 Saved: 20191112T073141_20191112T073144_T38RPS_label.tif

 Saved: 20200316T072639_20200316T073813_T38RPS_label.tif
 Saved: 20200415T072609_20200415T073422_T38RPR_label.tif

 Saved: 20181030T081041_20181030T082011_T37SDU_label.tif

 Saved: 20190324T080609_20190324T081236_T37SDU_label.tif

 Saved: 20190329T080